# Analysing PGD — variants, optimizers, and the GKDE study

The attack bench showed PGD working; this notebook takes PGD apart. Four variants (PGD, PGD2, PGD-min, PGD-min2) are compared norm by norm; then the optimizer itself is questioned — plain gradient steps vs **Adam**, with a study of Adam's β₁ — and the loss curves are plotted head-to-head. The second half studies **GKDE**: adding a kernel-density pull toward benign samples, across kernels, bandwidths and penalty factors, and ends with PGD-min measured against the RBF (Gaussian-kernel) model.

The originals ran every parameter combination; here each configuration appears once — the grid pattern repeats. **Where this fits:** Section 03 attacks the detectors built in `02-detectors/`, under the manipulation rules from `01-data-and-features/manipulation_maps.ipynb`. The defended models these attacks are later measured against live in `04-defenses/`, and the final, organised measurements are in `06-thesis-evaluation/`.

*Consolidated from `analyzing_PGD__2.ipynb` (the expanded version, as the spine) and `analyzing_PGD_.ipynb` (originally in `mal_adv3`). Outputs cleared.*

## Step 1 — Setup: data, the three trained models, the RBF model

# Loading data

In [ ]:
#RBF_models

download_links = [
                  'https://drive.google.com/uc?id=1-8lJXLdAl_4NdDwzw9kFML0aiOCTrI9f',
                  'https://drive.google.com/uc?id=1-OHACrNCt0yKBbdqQPVfNZcjKt5_jxKD',
                  'https://drive.google.com/uc?id=1-KeXJXtU1_6m9JOhormeVwigy0myX3HL',
                  'https://drive.google.com/uc?id=1-13RDdZqnrNkdHg3D8PC5KI0CZREwlsz',
                  'https://drive.google.com/uc?id=1-8LjsCdzKh6asxCFsYLiQZbSEXXKSQBP',

]

import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_DNN_drebin_best.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM_weightedLoss.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py'
]

In [ ]:
import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import pickle
import random
import time

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from adverserial_attacks_functions import *


torch.manual_seed(0)

In [ ]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    #os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

set_seed(42)

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()
print('the proportion of malwares : ', n_mal/(n_mal+n_ben))

# Combine features and labels into datasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# Define the DataLoader for training, validation, and test sets
batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Clear unnecessary variables
del train_dataset, val_dataset, test_dataset, y_train, y_val, y_test

In [ ]:
model_DNN = MalwareDetectionModel().to(device)
# Load model parameters
model_DNN.load_state_dict(torch.load('model_DNN_drebin_best.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM_weightedLoss = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM_weightedLoss.load_state_dict(torch.load('model_AT_rFGSM_weightedLoss.pth', map_location=torch.device(device)))

In [ ]:
model_evaluation(model_DNN, test_loader, device)

In [ ]:
model_evaluation(model_AT_rFGSM,test_loader,device)

In [ ]:
model_evaluation(model_AT_rFGSM_weightedLoss,test_loader,device)

In [ ]:
class RBFModel(nn.Module):
    def __init__(self, hidden_dim, output_dim, init_centers, init_sigmas, kernel):
        super(RBFModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.centers = nn.Parameter(torch.Tensor(init_centers))
        self.sigmas = nn.Parameter(torch.Tensor(init_sigmas))
        self.kernel = kernel
        # Linear layer for output
        self.linear = nn.Linear(hidden_dim, output_dim)

    def gaussian(self, x, c, sigma):
        return torch.exp(-torch.sum((x[:, None, :] - c) ** 2, dim=-1) / (2 * sigma ** 2))

    def laplacian(self, x, c, sigma):
        return torch.exp(-torch.sum(torch.abs(x[:, None, :] - c) , dim=-1) / sigma)

    def forward(self, x):
      if self.kernel == 'gaussian':
        radial_out = self.gaussian(x, self.centers, self.sigmas)
      elif self.kernel == 'laplacian':
        radial_out = self.laplacian(x, self.centers, self.sigmas)
      else:
        raise ValueError("Invalid kernel type. Choose 'gaussian' or 'laplacian'.")

      output = self.linear(radial_out.to(torch.float32))
      return output

In [ ]:
non_removal_features = False
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 10000))
model_gaussian_1000 = RBFModel(1000, 2, all_centers, [sigma], kernel)
model_gaussian_1000 = model_gaussian_1000.to(device)

# Load the model state dictionary
model_gaussian_1000.load_state_dict(torch.load('/content/best_model_gaussian_1000.pth', map_location=torch.device(device)))

In [ ]:
non_removal_features = True
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 1144))
model_gaussian_1000_nonremoval = RBFModel(1000, 2, all_centers, [sigma], kernel)
model_gaussian_1000_nonremoval = model_gaussian_1000_nonremoval.to(device)

# Load the model state dictionary
model_gaussian_1000_nonremoval.load_state_dict(torch.load('/content/best_model_gaussian_1000_nonremoval.pth', map_location=torch.device(device)))

In [ ]:
for x,y in test_loader:
  print(x.shape)
  print(y.shape)
  break

bens = x[y.squeeze()==0]
bens_y = y[y.squeeze()==0]
print(bens.shape)

mals = x[y.squeeze()==1]
mals_y = y[y.squeeze()==1]
print(mals.shape)

## Step 2 — The four PGD variants

#analyzing PGD

In [ ]:
x_input = mals.to(device)
# Boundary calculations
scaled_max_extended = torch.tensor(torch.maximum(insertion_array.float(), x_input),dtype=torch.float32)

scaled_min_extended = torch.tensor(torch.minimum((1. - removal_array.float()), x_input),dtype=torch.float32)

print(x_input[0,:20])
print(insertion_array[:20])
print(removal_array[:20])

print(scaled_min_extended[0,:20])
print(scaled_max_extended[0,:20])

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done

In [ ]:
def pgd(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Boundary calculations
    scaled_max_extended = torch.maximum(insertion_array.float(), x)
    scaled_min_extended = torch.minimum((1. - removal_array.float()), x)

    # Multi-step PGD
    for t in range(k):

        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=scaled_min_extended, max=scaled_max_extended)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of updated features',torch.mean(diff))

    return x_next

In [ ]:
def pgd_min(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Boundary calculations
    scaled_max_extended = torch.maximum(insertion_array.float(), x)
    scaled_min_extended = torch.minimum((1. - removal_array.float()), x)

    # Multi-step PGD
    for t in range(k):

        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=scaled_min_extended, max=scaled_max_extended)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of updated features',torch.mean(diff))

    return x_next

In [ ]:
def pgd2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

## Step 3 — The norm grid: L1

norm l1

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd2, device, **attack_params)

## Step 4 — The norm grid: L2

norm l2

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd2, device, **attack_params)

## Step 5 — The norm grid: L∞

norm linf


In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd2, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 250, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd2, device, **attack_params)

## Step 6 — The Adam variant, and the β₁ study

# ADAM optimization

In [ ]:
def pgd_min2_adam(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Adam optimizer variables
    m = torch.zeros_like(x)
    v = torch.zeros_like(x)
    beta1 = 0.9
    beta2 = 0.999
    epsilon_adam = 1e-8



    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion

        m = beta1 * m + (1 - beta1) * gradients
        v = beta2 * v + (1 - beta2) * (gradients ** 2)
        m_hat = m / (1 - beta1 ** (t+1))
        v_hat = v / (1 - beta2 ** (t+1))
        #print(torch.abs(v_hat).sum())
        gradients = m_hat / (torch.sqrt(v_hat) + epsilon_adam)

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
            #print(l2norm[0])
        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd_min2_adam, device, **attack_params)

In [ ]:
#beta1 = 0.75
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'free', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd_min2_adam, device, **attack_params)

In [ ]:
#beta1 = 0.75
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 200, 'step_length': 0.5, 'norm': 'free', 'round_threshold':0.9, 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd_min2_adam, device, **attack_params)

In [ ]:
#beta=0.75
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, pgd_min2_adam, device, **attack_params)

## Step 7 — Head-to-head: PGD-min2 vs PGD-min2-Adam loss curves

#PGD_min2 VS PGD_min2_adam

In [ ]:
def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next,loss_steps_i,loss_steps_d

In [ ]:
def pgd_min2_adam(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',beta1=0.9, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Adam optimizer variables
    m = torch.zeros_like(x)
    v = torch.zeros_like(x)
    #beta1 = 0.9
    beta2 = 0.999
    epsilon_adam = 1e-8

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion

        m = beta1 * m + (1 - beta1) * gradients
        v = beta2 * v + (1 - beta2) * (gradients ** 2)
        m_hat = m / (1 - beta1 ** (t+1))
        v_hat = v / (1 - beta2 ** (t+1))
        #print(torch.abs(v_hat).sum())
        gradients = m_hat / (torch.sqrt(v_hat) + epsilon_adam)

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
            #print(l2norm[0])
        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next,loss_steps_i,loss_steps_d

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_losses(loss1, loss2, loss3, loss4):
    """
    Plots three lists of loss values against iterations using seaborn.

    Parameters:
    loss1 (list): First list of loss values.
    loss2 (list): Second list of loss values.
    loss3 (list): Third list of loss values.
    loss4 (list): Third list of loss values.
    """
    # Create a list of iterations based on the length of the loss lists
    iterations = list(range(1, len(loss1) + 1))

    # Create a DataFrame
    data = pd.DataFrame({
        'Iteration': iterations * 4,
        'Loss': loss1 + loss2 + loss3 + loss4,
        'Type': ['Loss1'] * len(loss1) + ['Loss2'] * len(loss2) + ['Loss3'] * len(loss3)+ ['Loss4'] * len(loss4)
    })

    # Create the plot
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=data, x='Iteration', y='Loss', hue='Type')
    plt.title('Loss vs. Iteration')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.legend(title='Loss Type')
    plt.grid(True)
    plt.show()

**norm l1**

In [ ]:
# step_length for l1 20 times bigger because it affect a small part of features, and finally updated same number of features
x_next1,loss_steps_i1,loss_steps_d1 = pgd_min2(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=1., norm='l1', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next2,loss_steps_i2,loss_steps_d2 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=1., norm='l1',beta1=0.9, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next2,loss_steps_i3,loss_steps_d3 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=1., norm='l1',beta1=0.95, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next2,loss_steps_i4,loss_steps_d4 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=1., norm='l1',beta1=0.99, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
plot_losses(loss_steps_d1,loss_steps_d2,loss_steps_d3,loss_steps_d4)

**norm l2**

In [ ]:
# step_length for l1 20 times bigger because it affect a small part of features, and finally updated same number of features
x_next1,loss_steps_i1,loss_steps_d1 = pgd_min2(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.5, norm='l2', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next1,loss_steps_i2,loss_steps_d2 = pgd_min2(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.5, norm='max', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next2,loss_steps_i3,loss_steps_d3 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.5, norm='l2',beta1=0.9, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next2,loss_steps_i4,loss_steps_d4 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.5, norm='max',beta1=0.9, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

**norm linf**

In [ ]:
# step_length for l1 20 times bigger because it affect a small part of features, and finally updated same number of features
x_next1,loss_steps_i1,loss_steps_d1 = pgd_min2(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next1,loss_steps_i2,loss_steps_d2 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.02, norm='linf',beta1=0.9, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next2,loss_steps_i3,loss_steps_d3 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.02, norm='linf',beta1=0.85, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next2,loss_steps_i4,loss_steps_d4 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.02, norm='linf',beta1=0.75, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
x_next2,loss_steps_i4,loss_steps_d4 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=10, step_length=0.1, norm='free',beta1=0.75, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
def pgd_min2_adam_normalized(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',beta1=0.9, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Adam optimizer variables
    m = torch.zeros_like(x)
    v = torch.zeros_like(x)
    #beta1 = 0.9
    beta2 = 0.999
    epsilon_adam = 1e-8

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion

        un_mod = torch.abs(x - x_var) < 1.
        gradients = gradients * un_mod

        # Get the maximum gradient of each sample
        #print(gradients.shape)
        max_values, _ = torch.max(torch.abs(gradients), dim=-1)
        #print(max_values[:10])
        #print(torch.linalg.norm(gradients, dim=-1, keepdim=True)[:10])
        #gradients = (gradients / (max_values.view(gradients.size(0),-1)+ 1e-20))
        gradients = (gradients / (torch.linalg.norm(gradients, dim=-1, keepdim=True)+ 1e-20))
        #print(torch.count_nonzero(gradients, dim=1)[:10])


        m = beta1 * m + (1 - beta1) * gradients
        v = beta2 * v + (1 - beta2) * (gradients ** 2)
        m_hat = m / (1 - beta1 ** (t+1))
        v_hat = v / (1 - beta2 ** (t+1))
        #print(torch.abs(v_hat).sum())
        gradients = m_hat / (torch.sqrt(v_hat) + epsilon_adam)

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
            #print(l2norm[0])
        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next,loss_steps_i,loss_steps_d

In [ ]:
x_next1,loss_steps_i1,loss_steps_d1 = pgd_min2(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=100, step_length=.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next2,loss_steps_i2,loss_steps_d2 = pgd_min2_adam(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=100, step_length=.02, norm='free',beta1=0.9, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next2,loss_steps_i3,loss_steps_d3 = pgd_min2_adam_normalized(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=100, step_length=.02, norm='free',beta1=0.9, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
plot_losses(loss_steps_d1,loss_steps_d2,loss_steps_d3)

## Step 8 — Head-to-head: PGD vs PGD2 vs PGD-min on single samples

# **`analysis`**

# PGD VS PGD2

In [ ]:
x = mals.to(device)

# Expand insertion_array and removal_array to match the batch size
expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
expanded_removal_array = removal_array.expand(x.shape[0], -1)

# Update insertion and removal arrays based on input x
insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

In [ ]:
print(insertion_array.sum())
print(insertion_array_updated.sum(dim=-1))

print(removal_array.sum())
print(removal_array_updated.sum(dim=-1))

In [ ]:
def pgd(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        #pos_insertion = (x_var <= 0.5) * 1 * insertion_array
        pos_insertion = (x_var <= 0.5) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.5) * 1 * removal_array
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / l2norm).float()
            #perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            #perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            #perturbation[torch.isnan(perturbation)] = 0.
            #perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
def pgd2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    #insertion_array_updated = torch.bitwise_or(insertion_array.to(torch.uint8), x.squeeze().to(torch.uint8) )
    #removal_array_updated = torch.bitwise_or(removal_array.to(torch.uint8), (1 - x.squeeze().to(torch.uint8)) )

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        #pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        pos_insertion = (x_var <= 0.999) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
            #perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            #perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            #perturbation[torch.isnan(perturbation)] = 0.
            #perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
i=13
adv3 = pgd(mals[i:i+1].to(torch.float32).to(device), mals_y[i:i+1].to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
i=13
adv3 = pgd2(mals[i:i+1].to(torch.float32).to(device), mals_y[i:i+1].to(device), model_AT_rFGSM, insertion_array, removal_array, k=200, step_length=.01, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

# PG2 VS PGD_min

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done


def pgd_min(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))


    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        pos_insertion = (x_var <= 0.999) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()
            #print(torch.abs(perturbation).sum())
            #print('torch.abs(perturbation).sum(dim=-1) : ',torch.abs(perturbation).sum(dim=-1))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            #print('l2norm ; ',l2norm)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        #print(torch.abs(x_next - torch.clamp(x_next + perturbation * step_length, min=0., max=1.)).sum())
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
i=13
adv3 = pgd2(mals[i:i+1].to(torch.float32).to(device), mals_y[i:i+1].to(device), model_AT_rFGSM, insertion_array, removal_array, k=100, step_length=.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
i=13
adv3 = pgd_min(mals[i:i+1].to(torch.float32).to(device), mals_y[i:i+1].to(device), model_AT_rFGSM, insertion_array, removal_array, k=100, step_length=.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

# PGD vs PGD2 vs PGD_min

In [ ]:
def pgd2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    #insertion_array_updated = torch.bitwise_or(insertion_array.to(torch.uint8), x.squeeze().to(torch.uint8) )
    #removal_array_updated = torch.bitwise_or(removal_array.to(torch.uint8), (1 - x.squeeze().to(torch.uint8)) )

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []
    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        loss_steps_i.append(loss.detach().item())
        loss_steps_d.append(criterion(y_model, torch.zeros_like(y.view(-1).long())).detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        #pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        pos_insertion = (x_var <= 0.999) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
            #perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            #perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            #perturbation[torch.isnan(perturbation)] = 0.
            #perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next,loss_steps_i,loss_steps_d

In [ ]:
def pgd(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    loss_steps_i = []
    loss_steps_d = []
    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))
        loss_steps_i.append(loss.detach().item())
        loss_steps_d.append(criterion(y_model, torch.zeros_like(y.view(-1).long())).detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data
        #print(torch.abs(gradients).sum())

        pos_insertion = (x_var <= 0.5) * 1 * insertion_array
        #pos_insertion = (x_var <= 0.5) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients
        #print('nonzero(grad4insertion)',torch.count_nonzero(grad4insertion))

        pos_removal = (x_var > 0.5) * 0 * removal_array
        grad4removal = (gradients < 0) * pos_removal * gradients
        #print('nonzero(grad4removal)',torch.count_nonzero(grad4removal))

        gradients = grad4removal + grad4insertion
        #gradients =  gradients * tensor
        #print(torch.abs(gradients).sum())
        #print('nonzero(gradients)',torch.count_nonzero(gradients))

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / l2norm).float()
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next,loss_steps_i,loss_steps_d

In [ ]:
x_next,loss_steps_i,loss_steps_d = pgd(mals[14:15].to(torch.float32).to(device), mals_y[13:14].to(device), model_AT_rFGSM, insertion_array, removal_array, k=100, step_length=.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
x_next,loss_steps_i,loss_steps_d  = pgd(mals[13:14].to(torch.float32).to(device), mals_y[13:14].to(device), model_AT_rFGSM, insertion_array, removal_array, k=1000, step_length=.002, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
def pgd2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    #insertion_array_updated = torch.bitwise_or(insertion_array.to(torch.uint8), x.squeeze().to(torch.uint8) )
    #removal_array_updated = torch.bitwise_or(removal_array.to(torch.uint8), (1 - x.squeeze().to(torch.uint8)) )

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []
    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        #loss_steps_i.append(loss.detach().item())
        #loss_steps_d.append(criterion(y_model, torch.zeros_like(y.view(-1).long())).detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        #pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        pos_insertion = (x_var <= 0.999) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next,loss_steps_i,loss_steps_d

In [ ]:
x_next,loss_steps_i,loss_steps_d = pgd2(mals[13:14].to(torch.float32).to(device), mals_y[13:14].to(device), model_AT_rFGSM, insertion_array, removal_array, k=1000, step_length=.002, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
x_next,loss_steps_i,loss_steps_d = pgd_min(mals[13:14].to(torch.float32).to(device), mals_y[13:14].to(device), model_AT_rFGSM, insertion_array, removal_array, k=1000, step_length=.002, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
def pgd(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Multi-step PGD
    for t in range(k):

        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data


        pos_insertion = (x_var <= 0.5) * 1 * insertion_array
        #pos_insertion = (x_var <= 0.5) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.5) * 0 * removal_array
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / l2norm).float()
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next, torch.mean(diff)

In [ ]:
x_next,mean_diff = pgd(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=50, step_length=.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
mean_diff

In [ ]:
def pgd2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    #insertion_array_updated = torch.bitwise_or(insertion_array.to(torch.uint8), x.squeeze().to(torch.uint8) )
    #removal_array_updated = torch.bitwise_or(removal_array.to(torch.uint8), (1 - x.squeeze().to(torch.uint8)) )

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        #pos_insertion = (x_var <= 0.999) * 1
        pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next,torch.mean(diff)

In [ ]:
x_next,mean_diff = pgd_min(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, insertion_array, removal_array, k=50, step_length=.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

## Step 9 — Success rates

In [ ]:
# Assuming you have model, test_loader, and device defined
iterations_list = list(range(30, 300, 10))  # Specific iteration counts from 50 to 100 with a step of 5
step_size = 0.02  # Step size for PGD

In [ ]:
# Run PGD attack and plot the success rates
attack_success_rates, mean_diffs = run_pgd_attack(model_AT_rFGSM, pgd, test_loader, device, iterations_list, step_size=step_size)
plot_attack_statistics(iterations_list, attack_success_rates, mean_diffs)

In [ ]:
# Assuming you have model, test_loader, and device defined
iterations_list = list(range(30, 310, 10))  # Specific iteration counts from 50 to 100 with a step of 5
step_size = 0.02  # Step size for PGD

In [ ]:
# Run PGD attack and plot the success rates
attack_success_rates, mean_diffs = run_pgd_attack(model_AT_rFGSM_weightedLoss, pgd, test_loader, device, iterations_list, step_size=step_size)
plot_attack_statistics(iterations_list, attack_success_rates, mean_diffs)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def adv_predict(test_loader, model, attack, device, **kwargs):
    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0
    all_diff = 0.
    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            with torch.enable_grad():
                pertb_mal_x, diff = attack(mal_x_batch, mal_y_batch, model, insertion_array, removal_array, is_report_loss_diff=False, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()
            all_diff += (diff * len(mal_y_batch))

    # Calculate success rate as 1 - (correct predictions under attack / total samples)
    attack_success_rate = 1 - (cor_ad_test / n_samples)
    mean_diff = (all_diff/ n_samples)
    return attack_success_rate, mean_diff

def run_pgd_attack(model, attack, test_loader, device, iterations_list, step_size=0.5):
    attack_success_rates = []
    mean_diffs = []
    for k in iterations_list:
        #print(f"Running PGD with {k} iterations...")
        success_rate, mean_diff = adv_predict(test_loader, model, attack, device, k=k, step_length=step_size, norm='linf')
        attack_success_rates.append(success_rate)
        mean_diffs.append(mean_diff.item())

    return attack_success_rates, mean_diffs


def plot_attack_statistics(iterations, success_rates, updated_features):
    """
    Plots the attack success rate and mean number of updated features based on the number of iterations.

    Parameters:
    - iterations: List of iteration counts.
    - success_rates: List of success rates.
    - updated_features: List of mean number of updated features.
    """
    # Create DataFrames for each plot
    data_success_rate = pd.DataFrame({
        'Iterations': iterations,
        'Success Rate': success_rates
    })

    data_updated_features = pd.DataFrame({
        'Iterations': iterations,
        'Updated Features': updated_features
    })

    # Plot using seaborn
    fig, ax1 = plt.subplots(figsize=(14, 6))

    sns.lineplot(data=data_success_rate, x='Iterations', y='Success Rate', marker='o', ax=ax1, label='Success Rate')
    ax1.set_title('PGD Attack Statistics vs. Number of Iterations')
    ax1.set_xlabel('Number of Iterations')
    ax1.set_ylabel('Success Rate')
    ax1.grid(True)
    ax1.legend(loc='upper left')

    ax2 = ax1.twinx()
    sns.lineplot(data=data_updated_features, x='Iterations', y='Updated Features', marker='o', color='orange', ax=ax2, label='Updated Features')
    ax2.set_ylabel('Mean Number of Updated Features')
    ax2.legend(loc='upper right')

    plt.show()

In [ ]:
# Run PGD attack and plot the success rates
attack_success_rates, mean_diffs = run_pgd_attack(model_AT_rFGSM_weightedLoss, pgd_min_adam, test_loader, device, iterations_list, step_size=step_size)
plot_attack_statistics(iterations_list, attack_success_rates, mean_diffs)

## Step 10 — Validating the updated insertion/removal arrays

# validating updated insertion and removal array
wirh considering insertion array on the run


```
pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
```



In [ ]:
def pgd_min(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))


    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        #pos_insertion = (x_var <= 0.999) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()
            #print(torch.abs(perturbation).sum())
            #print('torch.abs(perturbation).sum(dim=-1) : ',torch.abs(perturbation).sum(dim=-1))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            #print('l2norm ; ',l2norm)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        #print(torch.abs(x_next - torch.clamp(x_next + perturbation * step_length, min=0., max=1.)).sum())
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
x = mals[13:14].to(torch.float32).to(device)
y = mals_y[13:14].to(device)
adv = pgd_min(x, y, model_AT_rFGSM, insertion_array, removal_array, k=1000, step_length=.01, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
print(x.sum())
print(adv.sum())

In [ ]:
insert = ((adv-x) > 0)
remove =  ((adv-x) < 0)
same = (adv != x)
print(insert.sum())
print(remove.sum())
print(same.sum())

In [ ]:
(insert * insertion_array_updated).sum()

In [ ]:
x = mals[3:4].to(torch.float32).to(device)
y = mals_y[3:4].to(device)
adv = pgd_min(x, y, model_AT_rFGSM, insertion_array, removal_array, k=1000, step_length=.01, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

## Step 11 — Conclusion runs

conclusion

In [ ]:
def pgd(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        pos_insertion = (x_var <= 0.5) * 1 * insertion_array
        #pos_insertion = (x_var <= 0.5) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.5) * 1 * removal_array
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / l2norm).float()
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
def pgd2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    #insertion_array_updated = torch.bitwise_or(insertion_array.to(torch.uint8), x.squeeze().to(torch.uint8) )
    #removal_array_updated = torch.bitwise_or(removal_array.to(torch.uint8), (1 - x.squeeze().to(torch.uint8)) )

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #print('loss : ',criterion(y_model, y.view(-1).long()))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        #pos_insertion = (x_var <= 0.999) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
            #perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            #perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            #perturbation[torch.isnan(perturbation)] = 0.
            #perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

## Step 12 — GKDE: kernels, bandwidths, and the cosine variant

# GKDE

In [ ]:
# Pre-select benign samples
benign_samples = []
mal_samples = []
for x_batch, y_batch in test_loader:
  benign_samples.append(x_batch[y_batch.squeeze() == 0])
  mal_samples.append(x_batch[y_batch.squeeze() == 1])

ben_x = torch.cat(benign_samples, dim=0).to(device)
mal_x = torch.cat(mal_samples, dim=0).to(device)

print(ben_x.shape)
print(mal_x.shape)

In [ ]:
def plot_min_difference(inputs, centers):
    title = 'Random Benigns'

    min_diffs = []
    with torch.no_grad():
        for x in inputs:
            min_diff = torch.inf
            for sample in centers:
                dif = torch.abs(sample - x).sum().item()
                if dif < min_diff:
                    min_diff = dif
            min_diffs.append(min_diff)

    min_diffs = np.array(min_diffs)

    # Print statistics
    print(f'{title} - Min: {np.min(min_diffs)}')
    print(f'{title} - Max: {np.max(min_diffs)}')
    print(f'{title} - Mean: {np.mean(min_diffs)}')

    # Plot histogram
    plt.figure(figsize=(15, 8))
    plt.hist(min_diffs, bins=120, edgecolor='black')
    plt.title(f'Distribution of Values - {title}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_min_difference(mal_x, ben_x[:500])

In [ ]:
import seaborn as sns

def gaussian_kernel(u, bandwidth):
    return torch.exp(-1. * torch.sum(u**2, dim=-1) / bandwidth**2)

def laplacian_kernel(u, bandwidth):
    return torch.exp(-1. * torch.sum(torch.abs(u), dim=-1) / bandwidth)

# Set up the range of values for u and bandwidth
u_values = torch.linspace(-40, 40, 100)
bandwidth = 10.

# Calculate the kernel contributions
gaussian_contributions = gaussian_kernel(u_values.unsqueeze(1), bandwidth)
laplacian_contributions = laplacian_kernel(u_values.unsqueeze(1), bandwidth)

# Create a DataFrame for plotting with Seaborn
import pandas as pd
df = pd.DataFrame({
    'u': u_values.numpy(),
    'Gaussian': gaussian_contributions.numpy(),
    'Laplacian': laplacian_contributions.numpy()
})

# Plot the kernels using Seaborn
sns.lineplot(x='u', y='value', hue='variable', data=pd.melt(df, ['u']))
plt.xlabel('u')
plt.ylabel('Kernel Contribution')
plt.title('Gaussian and Laplacian Kernels')
plt.show()

In [ ]:
# Multidimensional KDE implementation
def KDE(x, data, bandwidth,kernel):
    """
    Compute the kernel density estimate (KDE) for given data points.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).
        bandwidth (float): Bandwidth parameter for the KDE.

    Returns:
        torch.Tensor: Density estimate at each point in x (shape: [num_samples,]).
    """
    n = data.shape[0]  # Number of data points
    d = x.shape[1]  # Dimensionality of the data

    # Convert bandwidth to tensor
    bandwidth_tensor = torch.tensor(bandwidth)

    # Calculate standardized distances for all data points
    u = torch.abs(x[:, None, :] - data)
    #print(u.shape)
    # Compute kernel contributions for all data points
    if kernel == 'gaussian':
        kernel_contributions = torch.exp(-1. * torch.sum(u**2, dim=-1) / bandwidth_tensor**2)
    else:
        kernel_contributions = torch.exp(-1. * torch.sum(u, dim=-1) / bandwidth_tensor)
        #kernel_contributions = (1./d) * torch.exp(-1. * torch.sum(u, dim=-1) / bandwidth_tensor)

    # Sum contributions across all data points
    estimate = torch.mean(kernel_contributions, dim=1)

    # Normalize the density estimate by the number of points and the bandwidth raised to the dimensionality
    #estimate /= (bandwidth_tensor ** d)

    return estimate

In [ ]:
bandwidths = [0.1,0.5,1.,2.,5.,10.,15.,20.,30.,40.,50.]
for bandwidth in bandwidths:
    estimate = KDE(bens[:50], ben_x[:500], bandwidth,'laplac')
    estimate2 = KDE(mals[:50], ben_x[:500], bandwidth,'laplac')
    print('******* bandwidth : ',bandwidth)
    print('mean KDE(bens) :',estimate.mean())
    print('mean KDE(mals) :',estimate2.mean())

***cosine similarity***

In [ ]:
import torch

def normalize_vectors(vectors, epsilon=1e-10):
    """
    Normalize vectors to unit length.

    Parameters:
        vectors (torch.Tensor): Vectors to be normalized (shape: [num_vectors, num_dimensions]).
        epsilon (float): Small value to avoid division by zero.

    Returns:
        torch.Tensor: Normalized vectors (shape: [num_vectors, num_dimensions]).
    """
    vectors = vectors.to(torch.float32)
    norms = vectors.norm(dim=1, keepdim=True)
    norms = torch.clamp(norms, min=epsilon)  # Ensure norms are at least epsilon
    return vectors / norms


def cosine_dissimilarity(x, data):
    """
    Compute the cosine dissimilarity between points in x and data.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).

    Returns:
        torch.Tensor: Cosine dissimilarity (shape: [num_samples, num_data_points]).
    """
    # Normalize the vectors
    x_norm = normalize_vectors(x)
    data_norm = normalize_vectors(data)

    # Compute cosine similarity
    cos_sim = torch.mm(x_norm, data_norm.t())

    # Convert cosine similarity to dissimilarity
    cos_dissimilarity = torch.abs(1 - cos_sim) #avoiding negative value so close to zero(like -1e-10)

    return cos_dissimilarity

In [ ]:
def KDE(x, data, bandwidth, dissimilarity='euclidean', kernel='gaussian'):
    """
    Compute the kernel density estimate (KDE) for given data points.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).
        bandwidth (float): Bandwidth parameter for the KDE.
        dissimilarity (str): The type of dissimilarity to use ('cosine' or 'euclidean').
        kernel (str): The type of kernel to use ('gaussian' is supported).

    Returns:
        torch.Tensor: Density estimate at each point in x (shape: [num_samples,]).
    """
    # Convert bandwidth to tensor
    bandwidth_tensor = torch.tensor(bandwidth)

    # Calculate dissimilarity for all data points
    if dissimilarity == 'cosine':
        u = cosine_dissimilarity(x, data)
    elif dissimilarity == 'euclidean':
        u = torch.abs(x[:, None, :] - data)

    # Compute kernel contributions for all data points
    if kernel == 'gaussian':
        if dissimilarity == 'cosine':
            kernel_contributions = torch.exp(-u**2 / bandwidth_tensor**2)
        elif dissimilarity == 'euclidean':
            kernel_contributions = torch.exp(-torch.sum(u**2, dim=-1) / bandwidth_tensor**2)
    else:
        if dissimilarity == 'cosine':
            kernel_contributions = torch.exp(-u / bandwidth_tensor)
        elif dissimilarity == 'euclidean':
            kernel_contributions = torch.exp(-torch.sum(u, dim=-1) / bandwidth_tensor)

    # Sum contributions across all data points
    estimate = torch.mean(kernel_contributions, dim=1)

    return estimate

In [ ]:
KDE(mal_x[:20], ben_x[:500], bandwidth=0.5, dissimilarity='cosine', kernel='gaussian')

In [ ]:
def pgd2_kde(x, y, model,centers, bandwidth, dissimilarity, penalty_factor, kernel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long()) +  penalty_factor * KDE(x, centers, bandwidth, dissimilarity,kernel)

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        loss = criterion(model(x_var), y.view(-1).long()) +  penalty_factor * KDE(x_var, centers, bandwidth, dissimilarity,kernel)

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
def pgd_min2_kde(x, y, model,centers, bandwidth, dissimilarity, penalty_factor, kernel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long())) -  penalty_factor * KDE(x, centers, bandwidth, dissimilarity,kernel)

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        loss = criterion(model(x_var), torch.zeros_like(y.view(-1).long())) -  penalty_factor * KDE(x_var, centers, bandwidth, dissimilarity,kernel)
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
pgd_min2_kde(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM,ben_x[:100], 0.5, 'cosine', 100,'gaussian', insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    if (attack ==  mimicry) or (attack ==  pgd2_kde) or (attack ==  pgd_min2_kde):
      # Pre-select benign samples
      benign_samples = []
      for x_batch, y_batch in test_loader:
        benign_samples.append(x_batch[y_batch.squeeze() == 0])

      ben_x = torch.cat(benign_samples, dim=0).to(device)

      # Select the centers
      selected_benigns = ben_x[:500]

      del benign_samples, ben_x   # Free up memory


    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            if attack == mimicry:
                pertb_mal_x = mimicry(selected_benigns, mal_x_batch, model, **kwargs)
            elif (attack == pgd2_kde):
                with torch.enable_grad():
                  pertb_mal_x = pgd2_kde(mal_x_batch, mal_y_batch, model, selected_benigns, **kwargs)
            elif (attack == pgd_min2_kde):
                with torch.enable_grad():
                  pertb_mal_x = pgd_min2_kde(mal_x_batch, mal_y_batch, model, selected_benigns, **kwargs)
            else :
                with torch.enable_grad():
                    pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

******* bandwidth :  10.0

mean KDE(bens) : 0.2336

mean KDE(mals) : 0.0764

In [ ]:
attack_params =  {'bandwidth': 10.,'penalty_factor': 100,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd2_kde, device, **attack_params)

In [ ]:
penalty_factors = [1,10,100,1000,1e4,1e5]
bandwidths = [0.1,0.5,1.,2.,3.,5.,10.,15.,20.,30.,40.]

for penalty_factor in penalty_factors:
  for bandwidth in bandwidths:
    print('*********','penalty_factor:',penalty_factor,'bandwidth:',bandwidth,'******************')

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, pgd2_kde, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, pgd2_kde, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, pgd2_kde, device, **attack_params)

In [ ]:
penalty_factors = [1000,1e4]
bandwidths = [0.1,0.5,1.,2.,3.,5.,10.]

for penalty_factor in penalty_factors:
  for bandwidth in bandwidths:
    print('*********','penalty_factor:',penalty_factor,'bandwidth:',bandwidth,'******************')

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, pgd_min2_kde, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, pgd_min2_kde, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, pgd_min2_kde, device, **attack_params)

## Step 13 — GKDE: Bray-Curtis, and the gradient study

In [ ]:
import torch

def normalize_vectors(vectors, epsilon=1e-15):
    """
    Normalize vectors to unit length.

    Parameters:
        vectors (torch.Tensor): Vectors to be normalized (shape: [num_vectors, num_dimensions]).
        epsilon (float): Small value to avoid division by zero.

    Returns:
        torch.Tensor: Normalized vectors (shape: [num_vectors, num_dimensions]).
    """
    norms = vectors.norm(dim=1, keepdim=True)
    norms = torch.clamp(norms, min=epsilon)  # Ensure norms are at least epsilon
    return vectors / norms

def cosine_dissimilarity(x, data):
    """
    Compute the cosine dissimilarity between points in x and data.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).

    Returns:
        torch.Tensor: Cosine dissimilarity (shape: [num_samples, num_data_points]).
    """
    # Normalize the vectors
    x_norm = normalize_vectors(x)
    data_norm = normalize_vectors(data)

    # Compute cosine similarity
    cos_sim = torch.mm(x_norm, data_norm.t())

    # Convert cosine similarity to dissimilarity
    cos_dissimilarity = torch.abs(1 - cos_sim)

    return cos_dissimilarity

def bray_curtis_distance(x, data):
    """
    Compute the Bray-Curtis distance between points in x and data.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).

    Returns:
        torch.Tensor: Bray-Curtis distance (shape: [num_samples, num_data_points]).
    """
    numerator = torch.abs(x[:, None, :] - data).sum(dim=-1)
    denominator = torch.abs(x[:, None, :] + data).sum(dim=-1)
    bray_curtis = numerator / denominator
    return bray_curtis

def KDE(x, data, bandwidth, dissimilarity='euclidean', kernel='gaussian'):
    """
    Compute the kernel density estimate (KDE) for given data points.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).
        bandwidth (float): Bandwidth parameter for the KDE.
        dissimilarity (str): The type of dissimilarity to use ('cosine', 'euclidean', or 'braycurtis').
        kernel (str): The type of kernel to use ('gaussian' is supported).

    Returns:
        torch.Tensor: Density estimate at each point in x (shape: [num_samples,]).
    """
    # Convert bandwidth to tensor
    bandwidth_tensor = torch.tensor(bandwidth)

    # Calculate dissimilarity for all data points
    if dissimilarity == 'cosine':
        u = cosine_dissimilarity(x, data)
    elif dissimilarity == 'euclidean':
        u = torch.abs(x[:, None, :] - data)
    elif dissimilarity == 'braycurtis':
        u = bray_curtis_distance(x, data)
    else:
        raise ValueError(f"Unsupported dissimilarity metric: {dissimilarity}")

    # Compute kernel contributions for all data points
    if kernel == 'gaussian':
        if dissimilarity in ['cosine', 'braycurtis']:
            kernel_contributions = torch.exp(-u**2 / bandwidth_tensor**2)
        elif dissimilarity == 'euclidean':
            kernel_contributions = torch.exp(-torch.sum(u**2, dim=-1) / bandwidth_tensor**2)
    else:
        if dissimilarity in ['cosine', 'braycurtis']:
            kernel_contributions = torch.exp(-u / bandwidth_tensor)
        elif dissimilarity == 'euclidean':
            kernel_contributions = torch.exp(-torch.sum(u, dim=-1) / bandwidth_tensor)

    # Sum contributions across all data points
    estimate = torch.mean(kernel_contributions, dim=1)

    return estimate

In [ ]:
kde_estimates_braycurtis = KDE(mal_x[:20], ben_x[:500], bandwidth=0.2, dissimilarity='braycurtis', kernel='gaussian')
print(kde_estimates_braycurtis)
kde_estimates_braycurtis = KDE(ben_x[500:520], ben_x[:500], bandwidth=0.2, dissimilarity='braycurtis', kernel='gaussian')
print(kde_estimates_braycurtis)

In [ ]:
def pgd2_kde(x, y, model,centers, bandwidth, dissimilarity, penalty_factor, kernel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long()) +  penalty_factor * KDE(x, centers, bandwidth, dissimilarity,kernel)

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    #loss_steps_i = []
    loss_steps_d = []
    # Multi-step PGD
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        loss = criterion(model(x_var), y.view(-1).long()) +  penalty_factor * KDE(x_var, centers, bandwidth, dissimilarity,kernel)
        #loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        #loss_steps_d.append(loss.mean().detach().item())
        print(criterion(model(x_var), y.view(-1).long()).detach())
        loss_steps_d.append(criterion(model(x_var), y.view(-1).long()).mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data


        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(gradients.sum(dim=1))

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_var), y.view(-1).long()).data +  penalty_factor * KDE(x_var, centers, bandwidth, dissimilarity,kernel)
    #loss_adv = criterion(model(x_next), y.view(-1).long()).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next,loss_steps_d

In [ ]:
def pgd_min2_kde(x, y, model,centers, bandwidth, dissimilarity, penalty_factor, kernel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long())) -  penalty_factor * KDE(x, centers, bandwidth, dissimilarity,kernel)

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    #loss_steps_i = []
    loss_steps_d = []
    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        loss = criterion(model(x_var), torch.zeros_like(y.view(-1).long())) -  penalty_factor * KDE(x_var, centers, bandwidth, dissimilarity,kernel)

        #loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        #loss_steps_d.append(loss.mean().detach().item())
        loss_steps_d.append(criterion(model(x_var), torch.zeros_like(y.view(-1).long())).mean().detach())
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next,loss_steps_d

In [ ]:
x_next,loss_steps_d = pgd2_kde(mals[:3].to(torch.float32).to(device), mals_y[:3].to(device), model_AT_rFGSM,ben_x[:500], 1., 'euclidean', 1000,'laplac', insertion_array, removal_array, k=50, step_length=0.5, norm='l2', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next,loss_steps_d2 = pgd2_kde(mals[:3].to(torch.float32).to(device), mals_y[:3].to(device), model_AT_rFGSM,ben_x[:500], 1., 'euclidean', 1,'laplac', insertion_array, removal_array, k=50, step_length=0.5, norm='l2', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
plot_losses(loss_steps_d, loss_steps_d2, loss_steps_d2, loss_steps_d2)

In [ ]:
x_next,loss_steps_d = pgd_min2_kde(mals[:3].to(torch.float32).to(device), mals_y[:3].to(device), model_AT_rFGSM,ben_x[:500], 1., 'euclidean', 100000,'laplac', insertion_array, removal_array, k=50, step_length=0.5, norm='l2', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next,loss_steps_d2 = pgd_min2_kde(mals[:3].to(torch.float32).to(device), mals_y[:3].to(device), model_AT_rFGSM,ben_x[:500], 1., 'euclidean', 1,'laplac', insertion_array, removal_array, k=50, step_length=0.5, norm='l2', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

#Study GKDE gradients

In [ ]:
def get_loss_kde(adv_x,y,model,benigns, bandwidth, penalty_factor):
    criterion = nn.CrossEntropyLoss(reduction='none')
    outputs = model(adv_x)
    ce = criterion(outputs, y.view(-1).long())
    #print('ce: ', ce)
    kde = KDE(adv_x, benigns, bandwidth)
    #print('kde : ', kde)
    loss_no_reduction = ce + penalty_factor * kde
    _, predicted = torch.topk(outputs, k=1)
    done = (predicted != y).squeeze()

    return loss_no_reduction, done


def gkde(x, y, model,bens, bandwidth, penalty_factor, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural, _ = get_loss_kde(x,y,model,bens, bandwidth, penalty_factor)

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Multi-step PGD
    for t in range(k):
        print('************** t ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        #y_model = model(x_var)

        criterion = nn.CrossEntropyLoss(reduction='none')
        outputs = model(x_var)
        ce_loss = criterion(outputs, y.view(-1).long())
        print('ce_loss: ', ce_loss)
        kde_loss = KDE(x_var, bens, bandwidth)
        print('kde_loss: ', kde_loss)
        ce_grad = torch.autograd.grad(ce_loss.mean(), x_var, retain_graph=True)[0].data
        kde_grad = torch.autograd.grad(kde_loss.mean(), x_var)[0].data
        print('ce_grad ',torch.abs(ce_grad).sum(dim=-1).detach())
        print('kde_grad ',torch.abs(kde_grad).sum(dim=-1).detach())
        penalty_factor = torch.abs(ce_grad).sum(dim=-1).detach()/(torch.abs(kde_grad).sum(dim=-1).detach()+ 1e-20)
        print('penalty_factor ',penalty_factor)

        if t > 5:
          decayed_penalty_factor = penalty_factor * (1 - t / k)
        else:
          decayed_penalty_factor = penalty_factor

        # Compute loss
        loss, _ = get_loss_kde(x_var,y,model,bens, bandwidth, decayed_penalty_factor)
        #loss,_ = get_loss_kde(x_var,y,model,bens, bandwidth, penalty_factor)
        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data
        pos_insertion = (x_var <= 0.5) * 1 * insertion_array
        #pos_insertion = (x_var <= 0.5) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.5) * 1 * removal_array
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / l2norm).float()
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x_next - x_var) <= 1e-6
            gradients = gradients * un_mod

            val, _ = torch.topk(torch.abs(gradients), 1)
            perturbation = (torch.abs(gradients) >= val.expand_as(gradients)).float() * torch.sign(gradients).float()
            # stop perturbing the examples that are successful to evade the victim
            _, done = get_loss_kde(x_next,y,model,bens, bandwidth, penalty_factor)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    outputs = model(x_next)
    loss_adv = criterion(outputs, y.view(-1).long())
    _, predicted = torch.topk(outputs, k=1)
    done = (predicted != y).squeeze()

    if is_report_loss_diff:
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
# Pre-select benign samples
benign_samples = []

for x_batch, y_batch in test_loader:
  benign_samples.append(x_batch[y_batch.squeeze() == 0])

ben_x = torch.cat(benign_samples, dim=0).to(device)

# Forward pass to get logits for benign samples
with torch.no_grad():  # No need for gradients
    outputs = model_AT_rFGSM(ben_x.to(torch.float32))

# Calculate softmax probabilities
probabilities = torch.softmax(outputs, dim=1)

# Sort indices based on probabilities of class 1 (assuming class 1 is the "positive" class)
sorted_indices = torch.argsort(probabilities[:, 1], descending=False)

# Select the top 500 high confidence benign samples
top_500_high_confidence_benign_samples = ben_x[sorted_indices[:500]]

del benign_samples, outputs, probabilities, ben_x  # Free up memory

In [ ]:
adv = gkde(mals.to(torch.float32).to(device), mals_y.to(device), model_AT_rFGSM, top_500_high_confidence_benign_samples,0.6,1., insertion_array, removal_array, k=100, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

## Step 14 — GKDE: the variants and the penalty-factor sweeps

# Different kinds of GKDE

In [ ]:
# Pre-select benign samples
benign_samples = []
for x_batch, y_batch in test_loader:
  benign_samples.append(x_batch[y_batch.squeeze() == 0])

ben_x = torch.cat(benign_samples, dim=0).to(device)

# Forward pass to get logits for benign samples
with torch.no_grad():  # No need for gradients
    outputs = model_AT_rFGSM(ben_x.to(torch.float32))

# Calculate softmax probabilities
probabilities = torch.softmax(outputs, dim=1)

# Sort indices based on probabilities of class 1 (assuming class 1 is the "positive" class)
sorted_indices = torch.argsort(probabilities[:, 1], descending=False)

# Select the top 1000 high confidence benign samples
top_1000_high_confidence_benign_samples = ben_x[sorted_indices[:1000]]

# Set to track indices of samples to be removed
removed_set = set()

# Step 1: Identify similar samples
for i in range(len(top_1000_high_confidence_benign_samples)):
    if i in removed_set:
        continue
    sample1 = top_1000_high_confidence_benign_samples[i]
    for j in range(i + 1, len(top_1000_high_confidence_benign_samples)):
        if j in removed_set:
            continue
        sample2 = top_1000_high_confidence_benign_samples[j]
        if torch.abs(sample1 - sample2).sum().item() < 5:
            removed_set.add(j)

# Step 2: Select samples that are not in removed_set
selected_samples = [sample for idx, sample in enumerate(top_1000_high_confidence_benign_samples) if idx not in removed_set]

# Step 3: Stack tensors along a new dimension
selected_benigns = torch.stack(selected_samples)

del benign_samples, outputs, probabilities, top_1000_high_confidence_benign_samples, selected_samples   # Free up memory

In [ ]:
# Multidimensional KDE implementation
def KDE(x, data, bandwidth,kernel):
    """
    Compute the kernel density estimate (KDE) for given data points.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).
        bandwidth (float): Bandwidth parameter for the KDE.

    Returns:
        torch.Tensor: Density estimate at each point in x (shape: [num_samples,]).
    """
    n = data.shape[0]  # Number of data points
    d = x.shape[1]  # Dimensionality of the data

    # Convert bandwidth to tensor
    bandwidth_tensor = torch.tensor(bandwidth)

    # Calculate standardized distances for all data points
    u = torch.abs(x[:, None, :] - data)

    # Compute kernel contributions for all data points
    if kernel == 'gaussian':
        kernel_contributions = 10. * torch.exp(-0.5 * torch.sum(u**2, dim=-1) / bandwidth**2)
    else:
        kernel_contributions = (1./d) * torch.exp(-1. * torch.sum(u, dim=-1) / bandwidth_tensor)

    # Sum contributions across all data points
    estimate = torch.mean(kernel_contributions, dim=1)

    # Normalize the density estimate by the number of points and the bandwidth raised to the dimensionality
    #estimate /= (bandwidth_tensor ** d)

    return estimate

In [ ]:
def get_loss_kde_old(adv_x,y,model,benigns, bandwidth, penalty_factor,kernel):
    criterion = nn.CrossEntropyLoss(reduction='none')
    outputs = model(adv_x)
    ce = criterion(outputs, y.view(-1).long())
    #print('ce: ', ce)
    kde = KDE(adv_x, benigns, bandwidth,kernel)
    #print('kde : ', kde)
    loss_no_reduction = ce + penalty_factor * kde
    return loss_no_reduction


def gkde_old(x, y, model,bens, bandwidth, penalty_factor, kernel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = get_x0(x.clone(), initial_rounding_threshold, is_sample)

    # Multi-step PGD
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)

        # Compute loss
        loss = get_loss_kde_old(x_var,y,model,bens, bandwidth, penalty_factor, kernel)
        #loss,_ = get_loss_kde(x_var,y,model,bens, bandwidth, penalty_factor)
        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data
        pos_insertion = (x_var <= 0.5) * 1 * insertion_array
        #pos_insertion = (x_var <= 0.5) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.5) * 1 * removal_array
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / l2norm).float()
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x_next - x_var) <= 1e-6
            gradients = gradients * un_mod

            val, _ = torch.topk(torch.abs(gradients), 1)
            perturbation = (torch.abs(gradients) >= val.expand_as(gradients)).float() * torch.sign(gradients).float()
            # stop perturbing the examples that are successful to evade the victim
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    outputs = model(x_next)
    loss_adv = criterion(outputs, y.view(-1).long()).data
    done = get_done(x_next, y, model)

    if is_report_loss_diff:
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    if (attack ==  gkde) or (attack ==  mimicry) or (attack ==  gkde_old) or (attack ==  gkde_old2):
      # Pre-select benign samples
      benign_samples = []
      for x_batch, y_batch in test_loader:
        benign_samples.append(x_batch[y_batch.squeeze() == 0])

      ben_x = torch.cat(benign_samples, dim=0).to(device)

      # Forward pass to get logits for benign samples
      with torch.no_grad():  # No need for gradients
          outputs = model_AT_rFGSM(ben_x.to(torch.float32))

      # Calculate softmax probabilities
      probabilities = torch.softmax(outputs, dim=1)

      # Sort indices based on probabilities of class 1 (assuming class 1 is the "positive" class)
      sorted_indices = torch.argsort(probabilities[:, 1], descending=False)

      # Select the top 500 high confidence benign samples
      top_1000_high_confidence_benign_samples = ben_x[sorted_indices[:1000]]

      # Set to track indices of samples to be removed
      removed_set = set()

      # Step 1: Identify similar samples
      for i in range(len(top_1000_high_confidence_benign_samples)):
          if i in removed_set:
              continue
          sample1 = top_1000_high_confidence_benign_samples[i]
          for j in range(i + 1, len(top_1000_high_confidence_benign_samples)):
              if j in removed_set:
                  continue
              sample2 = top_1000_high_confidence_benign_samples[j]
              if torch.abs(sample1 - sample2).sum().item() < 5:
                  removed_set.add(j)

      # Step 2: Select samples that are not in removed_set
      selected_samples = [sample for idx, sample in enumerate(top_1000_high_confidence_benign_samples) if idx not in removed_set]

      # Step 3: Stack tensors along a new dimension
      selected_benigns = torch.stack(selected_samples)

      del benign_samples, outputs, probabilities, ben_x, top_1000_high_confidence_benign_samples, selected_samples   # Free up memory


    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            if attack == mimicry:
                pertb_mal_x = mimicry(top_500_high_confidence_benign_samples, mal_x_batch, model, **kwargs)
            elif (attack == gkde):
                with torch.enable_grad():
                  pertb_mal_x = gkde(mal_x_batch, mal_y_batch, model, selected_benigns, **kwargs)
            elif (attack == gkde_old):
                with torch.enable_grad():
                  pertb_mal_x = gkde_old(mal_x_batch, mal_y_batch, model, selected_benigns, **kwargs)
            elif (attack == gkde_old2):
                with torch.enable_grad():
                  pertb_mal_x = gkde_old2(mal_x_batch, mal_y_batch, model, selected_benigns, **kwargs)
            else :
                with torch.enable_grad():
                    pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
# gkde_old(maximize loss & default removal insertion arrays), adv_predict(top benigns), laplacian

penalty_factors = [1,10,100,1000,1e4,1e5]
bandwidths = [1.,2.,3.,5.,10.,15.,20.,30.,40.]

for penalty_factor in penalty_factors:
  for bandwidth in bandwidths:
    print('*********','penalty_factor:',penalty_factor,'bandwidth:',bandwidth,'******************')

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde_old, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde_old, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde_old, device, **attack_params)

In [ ]:
# gkde_old2(maximize loss & updated removal insertion arrays), adv_predict(top benigns), laplacian

penalty_factors = [1,10,100,1000,1e4,1e5]
bandwidths = [1.,2.,3.,5.,10.,15.,20.,30.,40.]

for penalty_factor in penalty_factors:
  for bandwidth in bandwidths:
    print('*********','penalty_factor:',penalty_factor,'bandwidth:',bandwidth,'******************')

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde_old2, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde_old2, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde_old2, device, **attack_params)

In [ ]:
def get_loss_kde(adv_x,y,model,benigns, bandwidth, penalty_factor,kernel):

    criterion = nn.CrossEntropyLoss()

    # Compute CE loss
    outputs = model(adv_x)
    ce = criterion(outputs, y)
    #print('ce ',ce)

    # Compute KDE loss
    kde = KDE(adv_x, benigns, bandwidth,kernel)
    #print('kde ',kde)

    # Combine the losses with the penalty factor
    loss = ce - penalty_factor * kde
    #print('loss ',loss)

    return loss


def gkde(x, y, model,bens, bandwidth, penalty_factor, kernel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',
        initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.

    :param x: Feature vector
    :param y: Ground truth labels
    :param model: Neural network model
    :param RBFModel: Gaussian model for KDE
    :param insertion_array: Array for insertion operations
    :param removal_array: Array for removal operations
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf', 'l2', 'l1')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    #target's class
    traget_labels = torch.zeros_like(y.view(-1).long())

    # Compute natural loss and penalty_factor
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), traget_labels)



    # Initialize starting point
    x_next = get_x0(x.clone(), initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    for t in range(k):
        x_var = x_next.clone().detach().requires_grad_(True)
        # Compute loss
        loss = get_loss_kde(x_var,traget_labels,model,bens, bandwidth, penalty_factor, kernel)
        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        #pos_insertion = (x_var <= 0.999) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion

        if norm == 'linf':
            perturbation = torch.sign(gradients).float()
        elif norm == 'l2':
            l2norm = torch.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
        elif norm == 'l1':
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1', 'l2', or 'linf' norm.")

        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    if random:
        round_threshold = torch.rand_like(x_next)
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_adv = criterion(model(x_next), traget_labels).data
    done = get_done(x_next, y, model)

    if is_report_loss_diff:
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size(0) * 100:.3f}%.")

    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
# fixed penalty factor

penalty_factors = [1,10,100,1000,1e4,1e6,1e8,1e10]
bandwidths = [1.,2.,3.,5.,10.,15.,20.,30.]

for penalty_factor in penalty_factors:
  for bandwidth in bandwidths:
    print('*********','penalty_factor:',penalty_factor,'bandwidth:',bandwidth,'******************')

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde, device, **attack_params)

In [ ]:
def get_loss_kde(adv_x,y,model,benigns, bandwidth, penalty_factor,kernel):

    criterion = nn.CrossEntropyLoss()

    # Compute CE loss
    outputs = model(adv_x)
    ce = criterion(outputs, y)
    #print('ce ',ce)

    # Compute KDE loss
    kde = KDE(adv_x, benigns, bandwidth,kernel)
    #print('kde ',kde)

    # Combine the losses with the penalty factor
    loss = ce - penalty_factor * kde
    #print('loss ',loss)

    return loss


def gkde(x, y, model,bens, bandwidth, penalty_factor, kernel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',
        initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.

    :param x: Feature vector
    :param y: Ground truth labels
    :param model: Neural network model
    :param RBFModel: Gaussian model for KDE
    :param insertion_array: Array for insertion operations
    :param removal_array: Array for removal operations
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf', 'l2', 'l1')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    #target's class
    traget_labels = torch.zeros_like(y.view(-1).long())

    # Compute natural loss and penalty_factor
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), traget_labels)



    # Initialize starting point
    x_next = get_x0(x.clone(), initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    for t in range(k):
        x_var = x_next.clone().detach().requires_grad_(True)
        decayed_penalty_factor = penalty_factor * (1 - t / k)

        # Compute loss
        loss = get_loss_kde(x_var,traget_labels,model,bens, bandwidth, decayed_penalty_factor, kernel)
        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        #pos_insertion = (x_var <= 0.999) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion

        if norm == 'linf':
            perturbation = torch.sign(gradients).float()
        elif norm == 'l2':
            l2norm = torch.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
        elif norm == 'l1':
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1', 'l2', or 'linf' norm.")

        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    if random:
        round_threshold = torch.rand_like(x_next)
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_adv = criterion(model(x_next), traget_labels).data
    done = get_done(x_next, y, model)

    if is_report_loss_diff:
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size(0) * 100:.3f}%.")

    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
penalty_factors = [1.,10,100,1000,1e4,1e5,1e6,1e7,1e8]
bandwidths = [1.,2.,3.,5.,10.,15.,20.,30.,40.]

for penalty_factor in penalty_factors:
  for bandwidth in bandwidths:
    print('*********','penalty_factor:',penalty_factor,'bandwidth:',bandwidth,'******************')

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde, device, **attack_params)

    attack_params =  {'bandwidth': bandwidth,'penalty_factor': penalty_factor,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
    adv_predict(test_loader, model_AT_rFGSM, gkde, device, **attack_params)

## Step 15 — Choosing the centres

**study the centers **

In [ ]:
# Pre-select benign samples
benign_samples = []
for x_batch, y_batch in test_loader:
  benign_samples.append(x_batch[y_batch.squeeze() == 0])

ben_x = torch.cat(benign_samples, dim=0).to(device)

# Forward pass to get logits for benign samples
with torch.no_grad():  # No need for gradients
    outputs = model_AT_rFGSM(ben_x.to(torch.float32))

# Calculate softmax probabilities
probabilities = torch.softmax(outputs, dim=1)

# Sort indices based on probabilities of class 1 (assuming class 1 is the "positive" class)
sorted_indices = torch.argsort(probabilities[:, 1], descending=False)

# Select the top 1000 high confidence benign samples
top_1000_high_confidence_benign_samples = ben_x[sorted_indices[:5000]]

# Set to track indices of samples to be removed
removed_set = set()

# Step 1: Identify similar samples
for i in range(len(top_1000_high_confidence_benign_samples)):
    if i in removed_set:
        continue
    sample1 = top_1000_high_confidence_benign_samples[i]
    for j in range(i + 1, len(top_1000_high_confidence_benign_samples)):
        if j in removed_set:
            continue
        sample2 = top_1000_high_confidence_benign_samples[j]
        if torch.abs(sample1 - sample2).sum().item() < 10.1:
            removed_set.add(j)

# Step 2: Select samples that are not in removed_set
selected_samples = [sample for idx, sample in enumerate(top_1000_high_confidence_benign_samples) if idx not in removed_set]

# Step 3: Stack tensors along a new dimension
selected_benigns = torch.stack(selected_samples)

print(selected_benigns.shape)
del benign_samples, outputs, probabilities, top_1000_high_confidence_benign_samples, selected_samples   # Free up memory

In [ ]:
top_447_high_confidence = ben_x[sorted_indices[:479]]
bens = ben_x[:479]

In [ ]:
top_50_high_confidence = ben_x[sorted_indices[:50]]

## Step 16 — PGD-min vs the RBF model

# PGD_min VS RBF

In [ ]:
model_AT_rFGSM(ben_x[sorted_indices[-5:]].to(torch.float32))

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done

def get_loss_rbf(adv_x: torch.Tensor, y: torch.Tensor, model: nn.Module, RBFModel: nn.Module, penalty_factor: torch.Tensor) -> torch.Tensor:

    criterion = nn.CrossEntropyLoss()

    # Compute CE loss
    outputs = model(adv_x)
    ce = criterion(outputs, y)
    #print('ce ',ce)

    # Compute KDE loss
    outputs_rbf = RBFModel(adv_x)
    kde = criterion(outputs_rbf, y)
    #print('kde ',kde)

    # Combine the losses with the penalty factor
    loss = ce + penalty_factor * kde
    #print('loss ',loss)

    return loss



def rbf(x, y, model, RBFModel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',
        initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.

    :param x: Feature vector
    :param y: Ground truth labels
    :param model: Neural network model
    :param RBFModel: Gaussian model for KDE
    :param insertion_array: Array for insertion operations
    :param removal_array: Array for removal operations
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf', 'l2', 'l1')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    #target's class
    traget_labels = torch.zeros_like(y.view(-1).long())

    # Compute natural loss and penalty_factor
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), traget_labels)
    kde = criterion(RBFModel(x), traget_labels)
    #penalty_factor = 0.
    penalty_factor = (loss_natural / kde).detach()

    # Initialize starting point
    x_next = get_x0(x.clone(), initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    for t in range(k):
        x_var = x_next.clone().detach().requires_grad_(True)
        decayed_penalty_factor = penalty_factor * (1 - t / k)

        # Compute loss
        loss = get_loss_rbf(x_var, traget_labels, model, RBFModel, decayed_penalty_factor)

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        pos_insertion = (x_var <= 0.999) * 1 * insertion_array_updated
        #pos_insertion = (x_var <= 0.999) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.001) * 1 * removal_array_updated
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion

        if norm == 'linf':
            perturbation = torch.sign(gradients).float()
        elif norm == 'l2':
            l2norm = torch.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()
        elif norm == 'l1':
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1', 'l2', or 'linf' norm.")

        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    if random:
        round_threshold = torch.rand_like(x_next)
    x_next = round_x(x_next, round_threshold=round_threshold)

    outputs = model(x_next)
    loss_adv = criterion(outputs, traget_labels).data
    done = get_done(x_next, y, model)

    if is_report_loss_diff:
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size(0) * 100:.3f}%.")

    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
# when penalty_factor = 0
attack_params =  {'RBFModel':model_gaussian_1000,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, rbf, device, **attack_params)

In [ ]:
# when penalty_factor = (loss_natural / kde).detach()
attack_params =  {'RBFModel':model_gaussian_1000,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, rbf, device, **attack_params)

In [ ]:
attack_params =  {'RBFModel':model_gaussian_1000,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, rbf, device, **attack_params)

In [ ]:
# when penalty_factor = (loss_natural / kde).detach()/2.
attack_params =  {'RBFModel':model_gaussian_1000,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, rbf, device, **attack_params)

In [ ]:
# when penalty_factor = (loss_natural / kde).detach()
attack_params =  {'RBFModel':model_gaussian_1000,'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_rFGSM, rbf, device, **attack_params)